# 04 – Live demo: a 70 m night temperature map of Istanbul in a few minutes

The two-step fusion (notebook 03) is trained once on all 51 clear ECOSTRESS nights; the trained model is
downloaded from the repository's release. For any night, this notebook fetches that night's MODIS LST from
Microsoft Planetary Computer (no account needed) and turns it into a 70 m map.

1. **A night with an ECOSTRESS overpass** (26 July 2021): the map is compared with ECOSTRESS.
   This night was part of the training set here; its held-out score is in notebook 03.
2. **A night without any usable ECOSTRESS image**: the case the method is built for.

Runtime on free Colab: about 3 minutes (most of it is installing packages).

## Setup

In [ ]:
import sys, subprocess, urllib.request
from pathlib import Path

REPO_URL = "https://github.com/KeremBasimc/istanbul-night-heat-island"
RELEASE = REPO_URL + "/releases/download/demo-v1/"
if "google.colab" in sys.modules or Path("/content").exists():
    ROOT = Path("/content/istanbul-night-heat-island")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

DEMO = ROOT / "data" / "demo"
DEMO.mkdir(parents=True, exist_ok=True)
for name in ["fusion_model.joblib", "static_2021.npz", "ecostress_2021-07-26.npz"]:
    if not (DEMO / name).exists():
        urllib.request.urlretrieve(RELEASE + name, DEMO / name)
    print(name, round((DEMO / name).stat().st_size / 1e6, 1), "MB")

## Load the trained model and the 70 m predictor layers

In [ ]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nhi import data, downscale as ds, model as M

coarse, fine = data.grids()
fusion = joblib.load(DEMO / "fusion_model.joblib")
static = dict(np.load(DEMO / "static_2021.npz"))
land = static.pop("land")
STATIC_DEV = M.static_with_deviations(static)
print("trained on", len(fusion["trained_on"]), "nights;", len(static), "predictor layers at 70 m,", land.shape)


def fetch_modis(night_date, prefer="terra"):
    """MODIS night LST at 980 m for the night that starts on night_date (live from Planetary Computer)."""
    passes = data.load_modis_night_passes(pd.Timestamp(night_date).date(), coarse)
    usable = {k: v for k, v in passes.items() if v["valid_frac"] >= 0.5}
    if not usable:
        raise ValueError(f"no MODIS night pass with at least half of the area cloud-free on {night_date}")
    platform = prefer if prefer in usable else next(iter(usable))
    p = usable[platform]
    h = p["clock_h"]
    local = pd.Timestamp(night_date) + pd.Timedelta(hours=h if h >= 12 else h + 24)
    return platform, p["lst"].values.astype("float32"), local


def night_map(modis_c, local, gap_h=0.0):
    """70 m night LST from one MODIS night: corrected 980 m blocks plus the 70 m detail."""
    night = {"modis_coarse": modis_c, "land": land, "local": local, "gap_h": gap_h, "eco": None,
             "layers": M.night_layers(STATIC_DEV, modis_c, local, gap_h)}
    return M.predict_fusion(fusion["block"], fusion["fine"], night, fusion["fine_names"], fusion["block_cols"])

## 1. A night with ECOSTRESS: 26 July 2021

In [ ]:
t0 = time.time()
ref = np.load(DEMO / "ecostress_2021-07-26.npz")
eco = ref["eco"]
platform, modis_c, local = fetch_modis("2021-07-26", prefer=str(ref["platform"]))
pred = night_map(modis_c, pd.Timestamp(str(ref["local"])), float(ref["gap_h"]))
print(f"MODIS {platform} fetched and downscaled in {time.time() - t0:.0f} s")

up = np.where(land, ds.upsample(modis_c), np.nan)
vmin, vmax = np.nanpercentile(eco, [2, 98])
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2), constrained_layout=True)
for ax, arr, title in [(axes[0], up, f"MODIS {platform.title()} 980 m"),
                       (axes[1], pred, "Fusion 70 m"),
                       (axes[2], eco, "ECOSTRESS 70 m (reference)")]:
    im = ax.imshow(arr, cmap="inferno", vmin=vmin, vmax=vmax)
    if arr is not eco:
        m = ds.metrics(arr, eco)
        title += f"\nbias-removed RMSE {m['crmse']:.2f} °C, r {m['r']:.2f}"
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="Land surface temperature (°C)", shrink=0.9)
plt.show()

## 2. A night without ECOSTRESS

ECOSTRESS has no usable image of Istanbul on this night; MODIS has. Change `NIGHT` to any summer 2021 date
(the predictor layers in this demo are those of 2021) and run the cell again.

In [ ]:
NIGHT = "2021-08-03"   # 2021 heat wave; no ECOSTRESS overpass that night

t0 = time.time()
platform, modis_c, local = fetch_modis(NIGHT)
pred = night_map(modis_c, local)
print(f"MODIS {platform} ({local:%d %b %Y %H:%M} local) fetched and downscaled in {time.time() - t0:.0f} s")

up = np.where(land, ds.upsample(modis_c), np.nan)
vmin, vmax = np.nanpercentile(pred, [2, 98])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
for ax, arr, title in [(axes[0], up, f"MODIS {platform.title()} 980 m, what exists every night"),
                       (axes[1], pred, "Fusion 70 m, the new night map")]:
    im = ax.imshow(arr, cmap="inferno", vmin=vmin, vmax=vmax)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="Land surface temperature (°C)", shrink=0.9)
fig.suptitle(f"Night of {pd.Timestamp(NIGHT):%d %B %Y}")
plt.show()

hot = pred > np.nanpercentile(pred, 95)
print(f"hottest 5 % of land: above {np.nanpercentile(pred, 95):.1f} °C, {hot.sum() * 0.0049:.0f} km²")